In [2]:
# !pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 78.5 MB/s eta 0:00:00


In [3]:
# !pip install faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 14.7 MB/s eta 0:00:00


In [4]:
from transformers import CLIPProcessor, CLIPModel, AutoProcessor, AutoModel
from PIL import Image
import requests
from datasets import load_dataset
from huggingface_hub import notebook_login
import torch
import faiss
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm

In [5]:
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
def dodaj_indeks(sample, idx):
    sample['id'] = idx
    return sample

In [10]:
print("Connecting to Caltech101...")
dataset_caltech = load_dataset("flwrlabs/caltech101", split="train", streaming=True, trust_remote_code=True)

dataset_caltech = dataset_caltech.map(dodaj_indeks, with_indices=True)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'flwrlabs/caltech101' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'flwrlabs/caltech101' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Connecting to Caltech101...


In [11]:
print("Connecting to ImageNet1k...")
dataset_imagenet = load_dataset("imagenet-1k", split="train", streaming=True, trust_remote_code=True)
dataset_imagenet = dataset_imagenet.map(dodaj_indeks, with_indices=True)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'imagenet-1k' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'imagenet-1k' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Connecting to ImageNet1k...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

In [ ]:
# print("Connecting to cc3m-wds...")
# dataset_cc3m = load_dataset("pixparse/cc3m-wds", split="train", streaming=True, trust_remote_code=True)
# print(type(dataset_cc3m))

# iterator = iter(dataset_cc3m)
# first_sample = next(iterator)

# print("\nDataset Labels:", first_sample['label'])
# first_sample['image']

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
model_id = "openai/clip-vit-base-patch16"
clip_model = CLIPModel.from_pretrained(model_id)
clip_processor = CLIPProcessor.from_pretrained(model_id)
clip_model.to(device)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [ ]:
model_id = "google/siglip-base-patch16-224"
siglip_model = AutoModel.from_pretrained(model_id)
siglip_processor = AutoProcessor.from_pretrained(model_id)
siglip_model.to(device)

config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/813M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json:   0%|          | 0.00/711 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

SiglipModel(
  (text_model): SiglipTextTransformer(
    (embeddings): SiglipTextEmbeddings(
      (token_embedding): Embedding(32000, 768)
      (position_embedding): Embedding(64, 768)
    )
    (encoder): SiglipEncoder(
      (layers): ModuleList(
        (0-11): 12 x SiglipEncoderLayer(
          (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attn): SiglipAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): SiglipMLP(
            (activation_fn): GELUTanh()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features

In [ ]:
model_id = "facebook/dinov2-base"
dino_model = AutoModel.from_pretrained(model_id)
dino_processor = AutoProcessor.from_pretrained(model_id)
dino_model.to(device)

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Dinov2Model(
  (embeddings): Dinov2Embeddings(
    (patch_embeddings): Dinov2PatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Dinov2Encoder(
    (layer): ModuleList(
      (0-11): 12 x Dinov2Layer(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attention): Dinov2Attention(
          (attention): Dinov2SelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
          )
          (output): Dinov2SelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (layer_scale1): Dinov2LayerScale()
        (drop_path): Identity()
        (norm2): LayerNorm((768,), eps=1e-06,

In [ ]:
def collect_embeddings(dataset_iterator, model, processor, device, max_samples=None):
    model.eval()

    all_embeddings = []
    all_labels = []

    print("Rozpoczynam ekstrakcję wektorów...")

    for i, sample in tqdm(enumerate(dataset_iterator), total=max_samples):
        # Przerywnik, jeśli chcemy przetestować kod na mniejszej próbce (np. max_samples=100)
        if max_samples and i >= max_samples:
            break

        image = sample['image']
        label = sample['label']

        # WAŻNE: Caltech101 ma różne formaty zdjęć (np. czarno-białe lub z kanałem alpha).
        # Modele wizyjne wymagają 3 kanałów (RGB).
        if image.mode != "RGB":
            image = image.convert("RGB")

        # 1. Przetwarzanie obrazu przez procesor
        inputs = processor(images=image, return_tensors="pt").to(device)

        # 2. Wyciągnięcie cech
        with torch.no_grad():
            # Niektóre modele (jak DINOv2) mogą wymagać bezpośredniego wywołania model(**inputs)
            # Używamy get_image_features lub standardowego wywołania
            outputs = model(**inputs) if not hasattr(model, 'get_image_features') else model.get_image_features(**inputs)

            # UNIWERSALNA EKSTRAKCJA TENSORA
            # Zabezpieczamy kod tak, aby działał i dla SigLIP, i dla DINOv2
            if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
                features = outputs.pooler_output
            elif hasattr(outputs, 'image_embeds') and outputs.image_embeds is not None:
                features = outputs.image_embeds
            elif hasattr(outputs, 'last_hidden_state'):
                # Fallback, gdyby model zwracał tylko stany ukryte (bierzemy token [CLS] czyli indeks 0)
                features = outputs.last_hidden_state[:, 0, :]
            else:
                features = outputs # Jeśli model jakimś cudem zwrócił czysty tensor

        # 3. KRYTYCZNY KROK: Normalizacja L2
        normalized_features = F.normalize(features, p=2, dim=1)

        # 4. Przenosimy na CPU i konwertujemy do numpy array (format wymagany przez FAISS)
        all_embeddings.append(normalized_features.cpu().numpy())
        all_labels.append(label)

    # Łączymy listę pojedynczych wektorów w jedną dużą macierz (N, 768)
    embeddings_matrix = np.vstack(all_embeddings).astype('float32')
    labels_array = np.array(all_labels)

    print(f"Zakończono! Kształt macierzy wektorów: {embeddings_matrix.shape}")

    return embeddings_matrix, labels_array

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import faiss
from tqdm.auto import tqdm

def collect_and_build_faiss_batched(dataset_iterator, model, processor, device, total_samples, batch_size=64):
    model.eval()

    # ZMIANA: Zaczynamy bez bazy FAISS. Zbudujemy ją, gdy model wypluje pierwszy wektor
    index = None

    all_labels = []
    batch_images = []
    batch_labels = []

    print(f"Rozpoczynam przetwarzanie wsadowe (batch_size={batch_size})...")

    for i, sample in tqdm(enumerate(dataset_iterator), total=total_samples):
        image = sample['image']
        if image.mode != "RGB":
            image = image.convert("RGB")

        batch_images.append(image)
        batch_labels.append(sample['label'])

        if len(batch_images) == batch_size or (i == total_samples - 1):

            inputs = processor(images=batch_images, return_tensors="pt").to(device)

            with torch.no_grad():
                outputs = model(**inputs) if not hasattr(model, 'get_image_features') else model.get_image_features(**inputs)

                if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
                    features = outputs.pooler_output
                elif hasattr(outputs, 'image_embeds') and outputs.image_embeds is not None:
                    features = outputs.image_embeds
                elif hasattr(outputs, 'last_hidden_state'):
                    features = outputs.last_hidden_state[:, 0, :]
                else:
                    features = outputs

            normalized_features = F.normalize(features, p=2, dim=1)
            wektory_numpy = normalized_features.cpu().numpy().astype('float32')

            # --- NOWY KLUCZOWY KROK ---
            # Jeśli to pierwsza paczka i nie mamy jeszcze bazy, sprawdzamy rozmiar i ją tworzymy
            if index is None:
                wymiar = wektory_numpy.shape[1] # Sprawdzamy, ile liczb ma wektor
                print(f"\n[INFO] Wykryto wymiar wektora: {wymiar}. Tworzę bazę FAISS...")
                index = faiss.IndexFlatIP(wymiar)
            # --------------------------

            index.add(wektory_numpy)

            all_labels.extend(batch_labels)

            batch_images = []
            batch_labels = []

        if i >= total_samples - 1:
            break

    print(f"Gotowe! Liczba wektorów w FAISS: {index.ntotal}")

    return index, np.array(all_labels)

In [ ]:
iterator_caltech = iter(dataset_caltech)

# siglip_embeddings, siglip_labels = collect_embeddings(
#     dataset_iterator=iterator_caltech,
#     model=siglip_model,
#     processor=siglip_processor,
#     device=device,
#     max_samples=1000
# )
siglip_faiss_index, siglip_labels = collect_and_build_faiss_batched(
    dataset_iterator=iterator_caltech,
    model=siglip_model,
    processor=siglip_processor,
    device=device,
    total_samples=1000,
    batch_size=64
)

Rozpoczynam przetwarzanie wsadowe (batch_size=64)...


  0%|          | 0/1000 [00:00<?, ?it/s]


[INFO] Wykryto wymiar wektora: 768. Tworzę bazę FAISS...
Gotowe! Liczba wektorów w FAISS: 1000


In [ ]:
iterator_caltech = iter(dataset_caltech)

# clip_embeddings, clip_labels = collect_embeddings(
#     dataset_iterator=iterator_caltech,
#     model=clip_model,
#     processor=clip_processor,
#     device=device,
#     max_samples=1000
# )
clip_faiss_index, clip_labels = collect_and_build_faiss_batched(
    dataset_iterator=iterator_caltech,
    model=clip_model,
    processor=clip_processor,
    device=device,
    total_samples=1000,
    batch_size=64
)

Rozpoczynam przetwarzanie wsadowe (batch_size=64)...


  0%|          | 0/1000 [00:00<?, ?it/s]


[INFO] Wykryto wymiar wektora: 512. Tworzę bazę FAISS...
Gotowe! Liczba wektorów w FAISS: 1000


In [ ]:
iterator_caltech = iter(dataset_caltech)

# dino_embeddings, dino_labels = collect_embeddings(
#     dataset_iterator=iterator_caltech,
#     model=dino_model,
#     processor=dino_processor,
#     device=device,
#     max_samples=1000
# )
dino_faiss_index, dino_labels = collect_and_build_faiss_batched(
    dataset_iterator=iterator_caltech,
    model=dino_model,
    processor=dino_processor,
    device=device,
    total_samples=1000,
    batch_size=64
)

Rozpoczynam przetwarzanie wsadowe (batch_size=64)...


  0%|          | 0/1000 [00:00<?, ?it/s]


[INFO] Wykryto wymiar wektora: 768. Tworzę bazę FAISS...
Gotowe! Liczba wektorów w FAISS: 1000
